## 1. Setup & Load Best Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
from LSTM_Price_Predictor import LSTMPricePredictor
from Data_Handling import get_data_auto

print("Libraries imported successfully!")
print(f"Validation Date: {datetime.now().strftime('%Y-%m-%d')}")

In [ ]:
# Load best architecture configuration from grid search
try:
    with open('best_architecture_config.json', 'r') as f:
        best_config = json.load(f)
    
    print("✓ Loaded best configuration from grid search:")
    print(f"  LSTM Units: {best_config['lstm_units']}")
    print(f"  Num Layers: {best_config['num_layers']}")
    print(f"  Dropout: {best_config['dropout']}")
    print(f"  Lookback: {best_config['lookback_window']} minutes ({best_config['lookback_window']/390:.1f} days)")
    print(f"  Grid Search Accuracy: {best_config['directional_accuracy']:.2f}%")
    
except FileNotFoundError:
    print("⚠️  best_architecture_config.json not found!")
    print("   Using default configuration. Run 01_Grid_Search_Architecture.ipynb first.")
    
    # Default fallback configuration
    best_config = {
        'lstm_units': 64,
        'num_layers': 2,
        'dropout': 0.2,
        'lookback_window': 585,
        'epochs': 30,
        'batch_size': 64,
        'learning_rate': 0.001,
        'prediction_horizon': 1
    }

## 2. Data Configuration (Monthly Update)

In [ ]:
# Symbol
SYMBOL = "SPY"
TIMEFRAME_UNIT = "Minute"
MULTIPLIER = 1

# Training: Last 6 months for robust model
# TODO: Update these dates monthly!
TRAIN_START = "2024-08-01"  # 6 months ago
TRAIN_END = "2025-01-31"    # Last completed month

# Testing: Current month
TEST_START = "2025-02-01"   # Current month
TEST_END = "2025-02-28"     # End of current month

# Multi-step prediction settings (recursive)
PREDICTION_HORIZONS = [1, 5, 10]  # Test 1-min, 5-min, 10-min ahead

print(f"Monthly Validation Configuration:")
print(f"  Symbol: {SYMBOL}")
print(f"  Training: {TRAIN_START} to {TRAIN_END} (6 months)")
print(f"  Testing: {TEST_START} to {TEST_END}")
print(f"  Prediction Horizons: {PREDICTION_HORIZONS} minutes")
print(f"  Note: Model trains on 1-step, recursively predicts {max(PREDICTION_HORIZONS)}-steps")

## 3. Load Data

In [ ]:
print("Loading training data (6 months)...")
train_data = get_data_auto(
    symbol=SYMBOL,
    start=TRAIN_START,
    end=TRAIN_END,
    timeframe_unit=TIMEFRAME_UNIT,
    multiplier=MULTIPLIER
)

print(f"\nLoading test data (current month)...")
test_data = get_data_auto(
    symbol=SYMBOL,
    start=TEST_START,
    end=TEST_END,
    timeframe_unit=TIMEFRAME_UNIT,
    multiplier=MULTIPLIER
)

print(f"\nData loaded:")
print(f"  Training: {len(train_data):,} bars ({train_data.index[0]} to {train_data.index[-1]})")
print(f"  Testing: {len(test_data):,} bars ({test_data.index[0]} to {test_data.index[-1]})")

## 4. Train Model with Best Configuration

In [ ]:
print("="*80)
print("TRAINING MODEL")
print("="*80)

# Initialize model with best parameters
model = LSTMPricePredictor(
    lookback_window=best_config['lookback_window'],
    prediction_horizon=best_config['prediction_horizon'],  # Train on 1-step
    lstm_units=best_config['lstm_units'],
    num_layers=best_config['num_layers'],
    dropout=best_config['dropout'],
    epochs=best_config['epochs'],
    batch_size=best_config['batch_size'],
    learning_rate=best_config['learning_rate'],
    prediction_mode='recursive',
    verbose=1
)

print(f"\nTraining with:")
print(f"  Architecture: {best_config['num_layers']} layers × {best_config['lstm_units']} units")
print(f"  Lookback: {best_config['lookback_window']} minutes")
print(f"  Training samples: {len(train_data):,}")
print(f"  Expected time: 10-15 minutes\n")

# Train
history = model.fit(
    train_data,
    validation_split=0.2,
    early_stopping_patience=10
)

print("\n✓ Training complete!")

## 5. Generate Predictions (Recursive Multi-Step)

In [ ]:
print("Generating predictions on test data...")
print(f"Note: Model predicts 1-step, then recursively extends to {max(PREDICTION_HORIZONS)} steps\n")

predictions = model.predict(test_data)

print(f"\nPredictions generated: {len(predictions):,} timestamps")
print(f"Period: {predictions.index[0]} to {predictions.index[-1]}")
print(f"\nAvailable prediction columns:")
print([col for col in predictions.columns if 'pred_price' in col])

## 6. Performance Evaluation - Multi-Horizon

In [ ]:
# Filter to trading hours (9:30 AM - 4:00 PM)
predictions['hour'] = predictions.index.hour
predictions['minute'] = predictions.index.minute

regular_hours_mask = (
    ((predictions['hour'] == 9) & (predictions['minute'] >= 30)) |
    ((predictions['hour'] >= 10) & (predictions['hour'] <= 15)) |
    ((predictions['hour'] == 16) & (predictions['minute'] == 0))
)
trading_hours_pred = predictions[regular_hours_mask].copy()

print("="*80)
print("MULTI-HORIZON PERFORMANCE ANALYSIS")
print("="*80)

# Analyze each prediction horizon
horizon_results = []

for horizon in PREDICTION_HORIZONS:
    pred_col = f'pred_price_{horizon}'
    
    if pred_col not in trading_hours_pred.columns:
        print(f"\n⚠️  {pred_col} not found. Model only predicts 1-step.")
        print(f"   To get {horizon}-min predictions, increase prediction_horizon in training.")
        continue
    
    # Calculate actual future price
    future_col = f'actual_future_{horizon}'
    trading_hours_pred[future_col] = trading_hours_pred['actual_price'].shift(-horizon)
    
    # Directional accuracy
    direction_correct = (
        np.sign(trading_hours_pred[pred_col] - trading_hours_pred['actual_price']) == 
        np.sign(trading_hours_pred[future_col] - trading_hours_pred['actual_price'])
    )
    
    valid_predictions = direction_correct.notna().sum()
    accuracy = direction_correct.mean() * 100
    
    horizon_results.append({
        'Horizon (min)': horizon,
        'Valid Predictions': valid_predictions,
        'Accuracy (%)': accuracy
    })

horizon_df = pd.DataFrame(horizon_results)
print("\n" + horizon_df.to_string(index=False))

print(f"\n{'='*80}")
print("INTERPRETATION:")
print(f"{'='*80}")
if len(horizon_results) > 1:
    best_horizon = horizon_df.loc[horizon_df['Accuracy (%)'].idxmax()]
    print(f"Best performing horizon: {int(best_horizon['Horizon (min)'])} minutes")
    print(f"  Accuracy: {best_horizon['Accuracy (%)']:.2f}%")
    
    if horizon_df.iloc[0]['Accuracy (%)'] > horizon_df.iloc[-1]['Accuracy (%)']:
        print(f"\n✓ Expected pattern: Accuracy degrades with longer horizons")
    else:
        print(f"\n⚠️  Unexpected: Longer horizons more accurate (trend persistence)")

## 7. Time-of-Day Analysis

In [ ]:
# Use 1-minute predictions for hourly analysis
pred_col = 'pred_price_1'
future_col = 'actual_future_1'

if future_col not in trading_hours_pred.columns:
    trading_hours_pred[future_col] = trading_hours_pred['actual_price'].shift(-1)

trading_hours_pred['direction_correct'] = (
    np.sign(trading_hours_pred[pred_col] - trading_hours_pred['actual_price']) == 
    np.sign(trading_hours_pred[future_col] - trading_hours_pred['actual_price'])
)

hourly_accuracy = trading_hours_pred.groupby('hour').agg({
    'direction_correct': 'mean',
    pred_col: 'count'
}).rename(columns={pred_col: 'count', 'direction_correct': 'accuracy'})
hourly_accuracy['accuracy'] *= 100

print("="*80)
print("PERFORMANCE BY HOUR OF DAY (1-minute predictions)")
print("="*80)
print(hourly_accuracy.to_string())

# Find best/worst hours
best_hour = hourly_accuracy['accuracy'].idxmax()
worst_hour = hourly_accuracy['accuracy'].idxmin()
print(f"\nBest hour: {best_hour}:00 ({hourly_accuracy.loc[best_hour, 'accuracy']:.2f}%)")
print(f"Worst hour: {worst_hour}:00 ({hourly_accuracy.loc[worst_hour, 'accuracy']:.2f}%)")

# Visualization
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(hourly_accuracy.index, hourly_accuracy['accuracy'], color='steelblue', alpha=0.8)
ax.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Random (50%)')
ax.set_xlabel('Hour of Day', fontsize=12, fontweight='bold')
ax.set_ylabel('Directional Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Model Performance by Hour', fontsize=14, fontweight='bold')
ax.set_xticks(range(9, 17))
ax.set_xticklabels(['9:30 AM', '10 AM', '11 AM', '12 PM', '1 PM', '2 PM', '3 PM', '4 PM'])
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Statistical Significance Testing

In [ ]:
from scipy import stats

print("="*80)
print("STATISTICAL SIGNIFICANCE TESTS")
print("="*80)

# Overall binomial test
n_correct = trading_hours_pred['direction_correct'].sum()
n_total = trading_hours_pred['direction_correct'].notna().sum()
accuracy = (n_correct / n_total) * 100

# Binomial test: H0: accuracy = 50%
binom_result = stats.binomtest(n_correct, n_total, p=0.5, alternative='greater')

print(f"\n1. OVERALL PERFORMANCE TEST")
print(f"   Accuracy: {accuracy:.2f}% ({n_correct}/{n_total})")
print(f"   P-value: {binom_result.pvalue:.6f}")
if binom_result.pvalue < 0.01:
    print(f"   Result: ✓✓ HIGHLY SIGNIFICANT (p < 0.01) - Model beats random!")
elif binom_result.pvalue < 0.05:
    print(f"   Result: ✓ SIGNIFICANT (p < 0.05) - Model likely beats random")
else:
    print(f"   Result: ✗ NOT SIGNIFICANT (p >= 0.05) - Cannot prove model beats random")

# Hour-by-hour significance
print(f"\n2. HOUR-BY-HOUR SIGNIFICANCE")
print("-" * 80)

hourly_significance = []
for hour in range(9, 17):
    hour_data = trading_hours_pred[trading_hours_pred['hour'] == hour]
    hour_correct = hour_data['direction_correct'].sum()
    hour_total = hour_data['direction_correct'].notna().sum()
    
    if hour_total > 0:
        hour_acc = (hour_correct / hour_total) * 100
        hour_pval = stats.binomtest(hour_correct, hour_total, p=0.5, alternative='greater').pvalue
        
        sig_text = ""
        if hour_pval < 0.01:
            sig_text = "***"
        elif hour_pval < 0.05:
            sig_text = "**"
        elif hour_pval < 0.10:
            sig_text = "*"
        
        hourly_significance.append({
            'Hour': hour,
            'Accuracy (%)': hour_acc,
            'N': hour_total,
            'P-Value': hour_pval,
            'Sig': sig_text
        })

sig_df = pd.DataFrame(hourly_significance)
print(sig_df.to_string(index=False))
print("\nLegend: *** p<0.01 | ** p<0.05 | * p<0.10")

## 9. TP/SL Optimization for Current Volatility

In [ ]:
print("="*80)
print("TP/SL OPTIMIZATION - Current Market Conditions")
print("="*80)

# Calculate returns
trading_hours_pred['predicted_return'] = (
    (trading_hours_pred['pred_price_1'] - trading_hours_pred['actual_price']) / 
    trading_hours_pred['actual_price']
) * 100

trading_hours_pred['actual_return'] = (
    (trading_hours_pred['actual_future_1'] - trading_hours_pred['actual_price']) / 
    trading_hours_pred['actual_price']
) * 100

# Test TP/SL combinations
tp_levels = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]  # % levels
sl_levels = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

tp_sl_results = []

for tp in tp_levels:
    for sl in sl_levels:
        # Simulate trades
        trades = trading_hours_pred[trading_hours_pred['direction_correct'].notna()].copy()
        
        # Determine outcome
        trades['tp_hit'] = np.abs(trades['actual_return']) >= tp
        trades['sl_hit'] = np.abs(trades['actual_return']) >= sl
        
        # Calculate P&L
        trades['pnl'] = 0.0
        
        # Correct predictions
        correct_mask = trades['direction_correct']
        trades.loc[correct_mask & trades['tp_hit'], 'pnl'] = tp
        trades.loc[correct_mask & ~trades['tp_hit'], 'pnl'] = trades.loc[correct_mask & ~trades['tp_hit'], 'actual_return']
        
        # Incorrect predictions
        incorrect_mask = ~trades['direction_correct']
        trades.loc[incorrect_mask & trades['sl_hit'], 'pnl'] = -sl
        trades.loc[incorrect_mask & ~trades['sl_hit'], 'pnl'] = trades.loc[incorrect_mask & ~trades['sl_hit'], 'actual_return']
        
        # Metrics
        win_rate = (trades['pnl'] > 0).mean() * 100
        avg_win = trades[trades['pnl'] > 0]['pnl'].mean() if (trades['pnl'] > 0).any() else 0
        avg_loss = trades[trades['pnl'] < 0]['pnl'].mean() if (trades['pnl'] < 0).any() else 0
        expected_value = trades['pnl'].mean()
        
        tp_sl_results.append({
            'TP (%)': tp,
            'SL (%)': sl,
            'Win Rate (%)': win_rate,
            'Avg Win (%)': avg_win,
            'Avg Loss (%)': avg_loss,
            'Expected Value (%)': expected_value,
            'N Trades': len(trades)
        })

tp_sl_df = pd.DataFrame(tp_sl_results)

# Find best configuration by EV
best_ev = tp_sl_df.loc[tp_sl_df['Expected Value (%)'].idxmax()]
print(f"\nBEST TP/SL CONFIGURATION (Max Expected Value):")
print(f"  TP: {best_ev['TP (%)']}%")
print(f"  SL: {best_ev['SL (%)']}%")
print(f"  Expected Value: {best_ev['Expected Value (%)']:.4f}% per trade")
print(f"  Win Rate: {best_ev['Win Rate (%)']:.2f}%")
print(f"  Avg Win: {best_ev['Avg Win (%)']:.4f}%")
print(f"  Avg Loss: {best_ev['Avg Loss (%)']:.4f}%")

# Heatmap
ev_pivot = tp_sl_df.pivot(index='SL (%)', columns='TP (%)', values='Expected Value (%)')
plt.figure(figsize=(10, 8))
sns.heatmap(ev_pivot, annot=True, fmt='.4f', cmap='RdYlGn', center=0, cbar_kws={'label': 'Expected Value (%)'})
plt.title('TP/SL Expected Value Heatmap', fontweight='bold', fontsize=14)
plt.xlabel('Take Profit (%)', fontweight='bold')
plt.ylabel('Stop Loss (%)', fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Summary & Trading Recommendations

In [ ]:
print("="*80)
print("MONTHLY VALIDATION SUMMARY")
print("="*80)

print(f"\n📅 Test Period: {TEST_START} to {TEST_END}")
print(f"\n📊 Overall Performance:")
print(f"   Directional Accuracy: {accuracy:.2f}%")
print(f"   Statistical Significance: p={binom_result.pvalue:.6f} {'✓ Significant' if binom_result.pvalue < 0.05 else '✗ Not significant'}")

if len(horizon_results) > 1:
    print(f"\n🎯 Multi-Horizon Results:")
    for result in horizon_results:
        print(f"   {result['Horizon (min)']}-min: {result['Accuracy (%)']:.2f}%")

print(f"\n⏰ Best Trading Hours:")
sig_hours = sig_df[sig_df['Sig'].str.contains('\*', regex=False)]
if len(sig_hours) > 0:
    for _, row in sig_hours.iterrows():
        print(f"   Hour {int(row['Hour'])}: {row['Accuracy (%)']:.2f}% {row['Sig']}")
else:
    print(f"   No statistically significant hours found")

print(f"\n💰 Optimal TP/SL:")
print(f"   Take Profit: {best_ev['TP (%)']}%")
print(f"   Stop Loss: {best_ev['SL (%)']}%")
print(f"   Expected Value: {best_ev['Expected Value (%)']:.4f}% per trade")

print(f"\n📈 Trading Recommendation:")
if accuracy > 52 and binom_result.pvalue < 0.05:
    print(f"   ✓ GREEN LIGHT - Model shows statistically significant edge")
    print(f"   ✓ Recommended to trade with optimized TP/SL")
elif accuracy > 50:
    print(f"   ⚠️  CAUTION - Model above 50% but not statistically significant")
    print(f"   ⚠️  Paper trade first or reduce position size")
else:
    print(f"   ✗ RED LIGHT - Model not performing above random")
    print(f"   ✗ Do not trade this month, re-evaluate parameters")

print(f"\n{'='*80}")
print(f"Next steps:")
print(f"1. {'✓ Proceed with trading' if accuracy > 52 and binom_result.pvalue < 0.05 else '✗ Skip trading this month'}")
print(f"2. Run this notebook again next month with updated dates")
print(f"3. Re-run grid search quarterly to adapt to market changes")
print(f"={'='*80}")